In [7]:
import pandas as pd
import os
import json

In [8]:
# 1. Asegurar carpetas
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../logs', exist_ok=True)

ruta_raw = 'dataset_PI.json'
ruta_log = 'logs/pipeline_log.csv'
ruta_processed = 'dataset_limpio.csv'

try:
    # 2. Cargar original
    with open(ruta_raw, 'r', encoding='utf-8') as f:
        df_original = pd.DataFrame(json.load(f))

    df_limpio = df_original.copy()

    # 3. Función de log
    def registrar_paso(paso, descripcion, df_actual, df_base=df_original, ruta=ruta_log):
        filas = len(df_actual)
        nulos = df_actual.isnull().sum().sum()
        retencion = round((filas / len(df_base)) * 100, 2)
        nuevo_registro = pd.DataFrame([{'Paso': paso, 'Descripción': descripcion, 'Filas': filas, 'Nulos': nulos, 'Retención (%)': retencion}])
        if not os.path.exists(ruta) or paso == 'Paso 0':
            nuevo_registro.to_csv(ruta, index=False, encoding='utf-8')
        else:
            nuevo_registro.to_csv(ruta, mode='a', header=False, index=False, encoding='utf-8')
        print(f"✔️ {paso} registrado correctamente.")

    # ==========================================
    # EJECUCIÓN DEL PIPELINE
    # ==========================================
    print("🚀 Iniciando proceso de limpieza...")
    
    registrar_paso('Paso 0', 'Carga inicial', df_limpio)

    # Paso 1: Edades
    df_limpio = df_limpio[(df_limpio['age'] >= 18) & (df_limpio['age'] <= 100)]
    registrar_paso('Paso 1', 'Filtro de edad (18-100)', df_limpio)

    # Paso 2: Planes
    df_limpio['subscription_plan'] = df_limpio['subscription_plan'].str.lower().str.strip()
    reemplazos_plan = {'estandar': 'Standard', 'estándar': 'Standard', 'std': 'Standard', 'standard': 'Standard', 'basicos': 'Basic', 'basico': 'Basic', 'básico': 'Basic', 'basic': 'Basic', 'premiun': 'Premium', 'premium': 'Premium'}
    df_limpio['subscription_plan'] = df_limpio['subscription_plan'].replace(reemplazos_plan).str.capitalize()
    registrar_paso('Paso 2', 'Estandarización de planes', df_limpio)

    # Paso 3: Minutos
    df_limpio['monthly_watch_time_mins'] = df_limpio['monthly_watch_time_mins'].fillna(df_limpio['monthly_watch_time_mins'].median())
    df_limpio = df_limpio[(df_limpio['monthly_watch_time_mins'] >= 0) & (df_limpio['monthly_watch_time_mins'] < 10000)]
    registrar_paso('Paso 3', 'Limpieza de minutos mensuales', df_limpio)

    # Paso 4: Tickets
    df_limpio = df_limpio[(df_limpio['customer_support_tickets'] >= 0) & (df_limpio['customer_support_tickets'] <= 20)]
    registrar_paso('Paso 4', 'Filtro de tickets extremos', df_limpio)

    # Paso 5: Países y Géneros
    mapeo_paises = {'brazil': 'Brasil', 'bra': 'Brasil', 'brasil': 'Brasil', 'arg': 'Argentina', 'argentina': 'Argentina', 'mex': 'México', 'mexico': 'México', 'col': 'Colombia', 'colombia': 'Colombia', 'uru': 'Uruguay', 'uruguay': 'Uruguay'}
    df_limpio['country'] = df_limpio['country'].str.lower().str.strip().replace(mapeo_paises)
    
    df_limpio['favorite_genre'] = df_limpio['favorite_genre'].str.lower().str.strip()
    reemplazos_genero = {'crime': 'Crimen', 'crimen': 'Crimen', 'action': 'Acción', 'accion': 'Acción', 'acción': 'Acción', 'documentary': 'Documental', 'Doc' : 'Documental', 'documental': 'Documental', 'comedy': 'Comedia', 'comedia': 'Comedia', 'sci-fi': 'Ciencia ficción', 'ciencia ficcion': 'Ciencia ficción', 'ciencia ficción': 'Ciencia ficción', 'romance': 'Romance'}
    df_limpio['favorite_genre'] = df_limpio['favorite_genre'].replace(reemplazos_genero).str.capitalize().fillna('No especificado')
    registrar_paso('Paso 5', 'Estandarización de países y géneros', df_limpio)

    # Paso 6: Fechas
    df_limpio['last_login_date'] = pd.to_datetime(df_limpio['last_login_date'], errors='coerce')
    df_limpio = df_limpio.dropna(subset=['last_login_date'])
    registrar_paso('Paso 6', 'Filtro de fechas inválidas', df_limpio)

    # GUARDADO FINAL
    print("💾 Guardando el archivo final en el disco duro...")
    df_limpio.to_csv(ruta_processed, index=False, encoding='utf-8')
    print("🎉 ¡TODO LISTO Y GUARDADO CON ÉXITO!")

except PermissionError:
    print("\n❌ ERROR DE PERMISOS: Tienes el archivo CSV abierto (probablemente en Excel). ¡Ciérralo y vuelve a correr la celda!")
except Exception as e:
    print(f"\n❌ ERROR INESPERADO: El código falló por esta razón: {e}")

🚀 Iniciando proceso de limpieza...
✔️ Paso 0 registrado correctamente.
✔️ Paso 1 registrado correctamente.
✔️ Paso 2 registrado correctamente.
✔️ Paso 3 registrado correctamente.
✔️ Paso 4 registrado correctamente.
✔️ Paso 5 registrado correctamente.
✔️ Paso 6 registrado correctamente.
💾 Guardando el archivo final en el disco duro...
🎉 ¡TODO LISTO Y GUARDADO CON ÉXITO!
